# Smart Grid Energy Management — Linear Regression

**Module:** Machine Learning  
**Algorithm:** Linear Regression (Supervised Learning)  
**Dataset:** Energy Consumption, Generation, Prices & Weather (Spain) — Kaggle  
**Dataset URL:** https://www.kaggle.com/datasets/nicholasjhana/energy-consumption-generation-prices-and-weather

---

## Problem Statement

The goal is to predict the **Actual Electricity Price** (€/MWh) using a standard Linear Regression model. This serves as a baseline to compare against more complex ensemble methods like Random Forest.

### Key Design Decisions
| Component | Design |
|---|---|
| **Target Variable** | `price actual` |
| **Predictor Features** | Load, Solar/Wind Generation, fossil gas, Hour, Month, Day of Week, Temp |
| **Model** | OLS Linear Regression |
| **Evaluation** | MAE, RMSE, and R² Score |

## 1. Imports & Style Configuration

In [ ]:
# Standard library
import os
import warnings
warnings.filterwarnings('ignore')

# Numerical / data
import numpy as np
import pandas as pd

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Styling
SEED = 42
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid')

print('Libraries imported successfully.')

## 2. Dataset Loading & Preprocessing

In [ ]:
ENERGY_FILE  = '../data/energy_dataset.csv'
WEATHER_FILE = '../data/weather_features.csv'

energy_df  = pd.read_csv(ENERGY_FILE,  parse_dates=['time'])
weather_df = pd.read_csv(WEATHER_FILE, parse_dates=['dt_iso'])

# Aggregate weather to hourly
weather_agg = (
    weather_df
    .rename(columns={'dt_iso': 'time'})
    .groupby('time')[['temp', 'wind_speed', 'clouds_all']]
    .mean()
    .reset_index()
)

# Handle timezones
energy_df['time'] = pd.to_datetime(energy_df['time'], utc=True).dt.tz_localize(None)
weather_agg['time'] = pd.to_datetime(weather_agg['time'], utc=True).dt.tz_localize(None)

# Merge
df = energy_df.merge(weather_agg, on='time', how='left')

# Feature Selection
core_cols = {
    'price actual': 'price',
    'total load actual': 'load',
    'generation solar': 'solar',
    'generation wind onshore': 'wind',
    'generation fossil gas': 'gas',
    'temp': 'temp'
}

df = df[['time'] + list(core_cols.keys())].rename(columns=core_cols)

# Cleaning
df.dropna(subset=['price', 'load'], inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

# Feature Engineering
df['hour'] = df['time'].dt.hour
df['month'] = df['time'].dt.month
df['day_of_week'] = df['time'].dt.dayofweek

print(f"Final Dataset Shape: {df.shape}")
df.head()

## 3. Training the Linear Regression Model

In [ ]:
features = ['load', 'solar', 'wind', 'gas', 'temp', 'hour', 'month', 'day_of_week']
X = df[features]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

print("Training Linear Regression model...")
lr = LinearRegression()
lr.fit(X_train, y_train)
print("Model training finished.")

## 4. Evaluation & Results

In [ ]:
y_pred = lr.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

results = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R2 Score'],
    'Value': [f"{mae:.4f} EUR/MWh", f"{rmse:.4f} EUR/MWh", f"{r2:.4f}"]
})

display(results)

## 5. Visual Analysis

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color='steelblue')
plt.plot([y.min(), y.max()], [y.min(), y.max()], '--r', lw=2)
plt.title("Linear Regression: Actual vs Predicted Prices")
plt.xlabel("Actual Price (€/MWh)")
plt.ylabel("Predicted Price (€/MWh)")
plt.show()